In [1]:
# use widgets as matplotlib backend
%matplotlib widget

# imports
from matplotlib import pyplot as plt
import os
import numpy
import os, sys
from time import time
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
import matplotlib as mpl

# import the file object for opening kongsberg files
# Note: function and library naming to be discussed
from themachinethatgoesping.echosounders import kmall
from themachinethatgoesping.echosounders import index_functions
from themachinethatgoesping.pingprocessing import filter_pings

#simplify creating figures
mpl.rcParams['figure.dpi'] = 100
close_plots = True

def create_figure(name, return_ax=True):
    if close_plots: plt.close(name)
    fig = plt.figure(name)
    fig.suptitle = name

    if return_ax:
        return fig, fig.subplots()
    return fig

In [2]:
# list of folders with kongsberg .all or .wcd files (subfolders will be scanned as well)

folder = "../../../unittest_data/"

# list raw data files
files,index = index_functions.find_files_and_index(folder, [".kmall","kmwcd"])
files.sort()
            
files.sort()
file_name = files[0]
print("files:      ", len(files))

Found 22 files
files:       22


In [3]:
import themachinethatgoesping as ping
print(ping.__file__)

/home/peurban/.local/share/mamba/envs/dev/lib/python3.14/site-packages/themachinethatgoesping/__init__.py


## Open all files
The function 
Notes: 
1. kmall.KMALLFileHandler(files) scanns and indexes all files and provides access to all files like a combined file stream
2. If a .all and a .wcd files with the same name (one .all and one .wcd) are added, they will be matched to a single file
3. It is not possible to add two .all or two .wcd with the same name, even if they are within different folders
4. Note: if the files are not sorted in time, the datagram packages will not be sorted by time either, however it isi simple to sort the pings at a later stage

In [4]:
# Index all files and initialize the file interfaces
# fm will be the accessor
fm = kmall.KMALLFileHandler(files,init=True)

#print some information about the files that where indexed
print(fm)

indexing files ⠐ 100% :00s<00m:00s] [..33285749584930.kmall (1/22)]                               
indexing files ⠠ 100% :00s<00m:00s] [..58516350552537.kmwcd (22/22)]                                
indexing files ⢀ 100% :00s<00m:00s] [Found: 502 datagrams in 22 files (11MB)]                                         
Initializing ping interface ⢀ 90% :00s<00m:00s] [Done]                                              
KMALLFileHandler
################
-
File infos 
-------------                 
- Number of loaded .kmall files: : 11       
- Number of loaded .kmwcd files: : 11       
- Total file size: :               11.28 MB 

 Detected datagrams 
^^^^^^^^^^^^^^^^^^^^ 
- timestamp_first:  26/07/2024 15:56:13.28 
- timestamp_last:   27/11/2025 14:14:45.23 
- Total:            502                    
- Datagrams [#MWC]: 58                     [M_WATER_COLUMN]
- Datagrams [#CHE]: 84                     [C_HEAVE]
- Datagrams [#SPE]: 33                     [S_POSITION_ERROR]
- Datagrams [#S

## How to access raw datagrams

In [5]:
# print all datagrams
print(fm.datagram_interface)

KMALLDatagramInterface
######################
-
Detected datagrams 
--------------------- 
- timestamp_first:  26/07/2024 15:56:13.28 
- timestamp_last:   27/11/2025 14:14:45.23 
- Total:            502                    
- Datagrams [#MWC]: 58                     [M_WATER_COLUMN]
- Datagrams [#CHE]: 84                     [C_HEAVE]
- Datagrams [#SPE]: 33                     [S_POSITION_ERROR]
- Datagrams [#SCL]: 34                     [S_CLOCK]
- Datagrams [#SKM]: 21                     [S_KM_BINARY]
- Datagrams [#CPO]: 35                     [C_POSITION]
- Datagrams [#SPO]: 70                     [S_POSITION]
- Datagrams [#IIP]: 22                     [I_INSTALLATION_PARAM]
- Datagrams [#IOP]: 22                     [I_OP_RUNTIME]
- Datagrams [#SVP]: 22                     [S_SOUND_VELOCITY_PROFILE]
- Datagrams [#SVT]: 20                     [S_SOUND_VELOCITY_TRANSDUCER]
- Datagrams [#MRZ]: 81                     [M_RANGE_AND_DEPTH]


In [6]:
""" 
Access some package and print it (note you should be able to use access 
all variables that are printed can be accessed using get_"varname"() function. 
Try get_ and use tab completition to see which functions are avaliable
"""

package = fm.datagram_interface.datagrams()[10]
print(package)

SClock
######
- bytes_datagram:      74         [bytes]
- datagram_identifier: #SCL       [S_CLOCK]
- datagram_version:    1          
- system_id:           40         
- echo_sounder_id:     2042       
- time_sec:            1764236334 [s]
- time_nanosec:        692454241  [ns]

 date/time 
-----------  
- timestamp: 1764.236e⁶   [s]
- date:      27/11/2025   [MM/DD/YYYY]
- time:      09:38:54.692 [HH:MM:SS]

 datagram content 
------------------                      
- bytes_content:          8                                 
- sensor_system:          5                                 
- sensor_status:          1                                 
- padding:                20039                             
- offset_sec:             0.000                             [s]
- clock_dev_pu_microsec:  -7545                             [µs]
  clock_data_from_sensor: $INZDA,093854.70,27,11,2025,,*72
                         

- bytes_datagram_check:   74                               [bytes

In [8]:
# You can also access packages by type
# print the first RuntimeParameters datagram
print(fm.datagram_interface.datagrams("I_OP_RUNTIME")[10])

IOpRuntime
##########
- bytes_datagram:      1302       [bytes]
- datagram_identifier: #IOP       [I_OP_RUNTIME]
- datagram_version:    0          
- system_id:           40         
- echo_sounder_id:     2042       
- time_sec:            1764241536 [s]
- time_nanosec:        858360583  [ns]

 date/time 
-----------  
- timestamp: 1764.242e⁶   [s]
- date:      27/11/2025   [MM/DD/YYYY]
- time:      11:05:36.858 [HH:MM:SS]

 datagram content 
------------------ 
- bytes_content:        1278 
- info:                 0    
- status:               0    
- bytes datagram check: 1302 [bytes]

 runtime_txt 
------------- 
- runtime_txt:  
              Sector coverage
              Max angle Port:      60.0
              Max angle Starboard: 60.0
              Max coverage Port:      50.0
              Max coverage Starboard: 50.0
              Angular coverage Mode: Auto
              Sector mode: Normal
              Beam spacing: 512 soundings
              
              Depth settings


In [9]:
# The datagrams function returns a containter like type. 
# Pacakges can be accessed using [index] but it is also possible to e.g. loop through the data using the functions typically used to lists.
# note that also standard slicing logic works

# loop through every 2nd package between index 2 and 5 and compute the average timestamp
timestamps = []
for p in tqdm(fm.datagram_interface.datagrams(skip_data=True)[2:5:2]): #skip_data speeds up looping because it ignores water column data samples
    timestamps.append(p.get_timestamp())

avg = np.nanmean(timestamps)

  0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
# ping has a bunch of more tools avaliable. E.g. convert unixtime to string and the other way around

# print the timestamp as text (use ping tools)
from themachinethatgoesping.tools.timeconv import unixtime_to_datestring

print(unixtime_to_datestring(np.nanmean(timestamps), format="%d-%m-%Y %H:%M:%S")) #try tab completition in the function to access the documentaiton

27-11-2025 09:38:54


In [11]:
# btw. print documentation is also possible like this
help(unixtime_to_datestring)

Help on nb_func in module themachinethatgoesping.tools_nanopy.timeconv:

unixtime_to_datestring = <nanobind.nb_func object>
    unixtime_to_datestring(unixtime: float, fractionalSecondsDigits: int = 0, format: str = '%z__%d-%m-%Y__%H:%M:%S') -> str

    Converts a UNIX timestamp to a formatted date/time string.
    Args:
        unixtime: UNIX timestamp as double (seconds since
                  1970-01-01T00:00:00Z)
        fractionalSecondsDigits: Number of digits for fractional seconds
                                 (default: 0)
        format: Format string (default: "%z__%d-%m-%Y__%H:%M:%S")

    Returns:
        Formatted date/time string

